In [ ]:
%%configure -f
{
  "defaultLakehouse": {
    "name": {
      "parameterName": "lakehouse_name",
      "defaultValue": "YourLakehouse"
    }
  }
}

> 🔗 **Session binding** — the `%%configure` cell above pins this run's default lakehouse **by
> name, before the session starts**. On a pipeline run the `lakehouse_name` **base parameter**
> feeds it — the same parameter the Parameters cell below receives, so one pipeline parameter
> drives both the session binding and `setup`'s assertion. Interactive users can instead simply
> attach a lakehouse in the portal; `defaultValue` is then what running the cell would bind, so
> keep it naming the lakehouse `lakehouse_name` asserts. Fail-visible either way: if no binding
> lands, the run blocks on OLAF's `no lakehouse attached` guard (and `setup` re-asserts the
> name) — it never writes anywhere unintended.

# OneLake Security — Master Notebook

Runs `olaf` through a full deployment, **one stage per cell**, with each stage's result deciding the
next. This is exactly what a Fabric / ADF pipeline does with one `notebook.run` activity per stage —
same sequence, same branching — just without the pipeline.

> ⚠️ **Before running:** attach the lakehouse you are securing. Every stage resolves its control
> tables and its target through whatever lakehouse is *attached*, so that attachment is the setting
> that matters. (The `lakehouse_name` parameter is read only by `setup`, which asserts the attached
> lakehouse is the one you named — a typo there is caught only when `run_setup = True`.) OLAF also refuses a lakehouse attached from a
> different workspace — run this from the workspace that owns it.

**`keep_unmanaged = False` is deliberate.** `apply` submits the config as the whole truth, so any
role the config does not declare is omitted from the payload — including one somebody added by hand in the portal, and including
`Default*` — a leftover `DefaultReader` reads every path and bypasses RLS/CLS, so leaving it in place
would quietly undo the whole config. That
that omission is the point: **the config is the only way to change access.** Editing roles outside the
framework does not survive the next run, by design — so do not flip this to `True` to make a run
feel safer.

---

## The flow

```
  setup ─▶ load_config ─▶ validate ─▶ generate ─▶ plan ─┬─▶ no changes ──────────────────▶ done
 ┈┈┈┈┈┈   ┈┈┈┈┈┈┈┈┈┈┈     ┈┈┈┈┈┈┈┈    ┈┈┈┈┈┈┈┈    ┈┈┈┈ │
  opt-in   workbook→       no writes   lock-file   diff └─▶ changes ─▶ gate ─▶ apply ─▶ verify
           control tables                                             ┈┈┈┈     ┈┈┈┈┈
                                                                  AUTO_APPROVE  the only
                                                                                live write
```

**Run it as often as you like.** Every stage is safe to repeat: `setup` is idempotent, `load_config`
is a full REPLACE from the workbook, `generate` skips when the config hash has not moved, and `plan`
finds no drift when the live state already matches. Re-running is the intended loop, not an
exceptional path.

> ⚠️ **Edited the workbook? Set `load_config = True` for that run.** It is `False` by default, and a
> run with it off never reads the workbook: the config table is unchanged, so `generate` skips, `plan`
> reports no drift, and the whole notebook goes green **without your edit ever reaching the vault**.
> The failure mode of forgetting is a success verdict, which is why it is called out here and not
> only at the stage.

| # | Stage | Writes | What it does |
|---|---|---|---|
| 1 | `setup` | control tables | Creates them if missing, and migrates columns after a framework upgrade. Off by default — set `run_setup = True` to run it. |
| 1b | `load_config` | config + member | Loads the authored workbook into the two author-owned control tables. A full REPLACE, so a row deleted in the workbook is deleted here. Writes no log row — this is authoring, not deployment. Off by default. |
| 2 | `validate` | **nothing** | Every rule `generate` runs, against the config. Collects *all* failures. |
| 3 | `generate` | mapping + CSV | Commits the config into the lock-file `plan` and `apply` read. |
| 4 | `plan` | log row | Diffs mapping against live. **No live change.** Sets `changed`. |
| 5 | `apply` | **live roles** | The only live write. Refuses unless a successful `plan` exists for this same **config_hash**, and the live state still matches it. |
| 6 | verify | nothing | Reads the live state back through the `OLAF` facade. |

**`validate` runs before `generate`, not after.** It is a dry-run of the very validation
`generate` performs, with zero writes — so it previews the commit, and previewing *after*
committing would be the wrong way round.

---

## Two ways to call OLAF, and the choice is load-bearing

This notebook uses **both**, on a rule:

| | used for | on a blocked stage |
|---|---|---|
| `notebookutils.notebook.run` | every stage whose refusal must **stop the run** — setup, validate, generate, plan, apply, rollback | **raises**, so the cell fails and the run stops |
| the `OLAF` facade | `configure` and `load_config`, which have no mode of their own · and the read-only tail (`show`, `trace`) where there is no refusal to gate on | returns a frame, **never raises on outcome** |

> The split is **not** "is it a mode?" — `show` and `trace` are modes (all eight are: setup, generate,
> validate, plan, apply, rollback, show, trace). The split is whether a refusal has to halt the run.
> A read stage has nothing to halt, so the frame is the more useful shape.

**Do not "simplify" this into an all-facade notebook.** The facade is the interactive surface and is
pinned by tests as never raising on outcome (`OLAF.generate()` on a config the member gate refuses
hands back a `blocked` frame). Drive the stages that way and a refused config prints a frame and the
run sails on into plan and apply — a fail-closed refusal turned into a silent continue. You would
have to re-check `OLAF.last_result["status"]` after every call and raise by hand, which is
`notebook.run` rewritten badly.

`notebook.run` also gives each stage its own session and its own parameters. For orchestration that
isolation is the feature, not the overhead — and it is what makes this notebook a faithful stand-in
for a pipeline with one activity per stage.

## How a stage reports back

`olaf` ends every mode with one envelope:

```
{ "mode"        the stage that ran          "data"         per-stage result keys
  "status"      success | skipped           "batch_id"     ties plan to apply
                blocked | error             "config_hash"  which config version
  "changed"     is there drift to apply     "message"      the one-line verdict     }
```

| status | What `olaf` does | What this notebook sees |
|---|---|---|
| `success` · `skipped` | `notebook.exit(envelope)` | `notebook.run` returns the JSON above |
| `blocked` · `error` | **raises** | the cell fails and the run stops here |

A failed stage still stops the notebook, so the calling pipeline takes its Failure path — but it is
not allowed to stop *silently*. Every cell goes through `run_stage` or its own try, which prints the
stage, the capped raise payload and this run's log rows **before** re-raising. The `if`s below branch
on *outcomes*; failures are reported and rethrown, never swallowed.

> It also means `blocked` and `error` never arrive as a *value*. Any variable that `json.loads`
> succeeded on is `success` or `skipped` — nothing else. The one stage where that costs something
> is `apply`, which has already written to the live role set by the time it can fail, so that cell
> prints the log row before letting the failure through.


## Parameters

This cell is tagged `parameters`: Fabric and papermill **replace it wholesale** at run time, so the
values below are only what a manual run falls back on. Keep it to bare assignments — anything else
written here is lost on a pipeline run.

| Parameter         | Default                                  | What it does                                                               |
|-------------------|------------------------------------------|----------------------------------------------------------------------------|
| `env`             | `"dev"`                                  | Tags every log row, so one estate's dev and prod audit trails stay apart.  |
| `auto_approve`    | `False`                                  | The gate. `True` lets `apply` run unattended — nobody reviews the omissions. |
| `keep_unmanaged`  | `False`                                  | `False` = config is the whole truth: a live role it omits is left out of the payload. |
| `rebuild`         | `False`                                  | `True` re-resolves wildcards against the live catalog even when the config has not changed. |
| `run_setup`       | `False`                                  | `True` after a framework upgrade — `setup` migrates control-table columns. |
| `load_config`     | `False`                                  | `True` re-reads the authored workbook into the config + member tables.     |
| `config_workbook` | `"Files/security/onelake_security.xlsx"` | The authored workbook, resolved on the attached lakehouse.                 |
| `batch_id`        | `""`                                     | Correlates this run's log rows. Empty = this notebook generates one.       |


In [ ]:
env = "dev"
lakehouse_name = "YourLakehouse"  # the attached lakehouse this run secures (setup asserts it)
auto_approve = False
keep_unmanaged = False
rebuild = False
run_setup = False
load_config = False
config_workbook = "Files/security/onelake_security.xlsx"
batch_id = ""

In [ ]:
import json
import re
import uuid
from collections import Counter

import notebookutils

# ── run constants ─────────────────────────────────────────────────────────────
#   OLAF_NB         the OLAF runtime notebook, by name, for notebook.run()
#   LAKEHOUSE       the attached lakehouse this run secures — from the lakehouse_name
#                   PARAMETER above, so a pipeline can set it per run (it was a hardcoded
#                   constant here through 1.0.x, invisible to Base parameters)
#   AUTO_APPROVE    True = run apply          ·  False = stop after plan
#   KEEP_UNMANAGED  False = config is whole truth ·  omits roles absent from the config
#   REBUILD         True = re-resolve wildcards against the live catalog · see cell 13
#   LOAD_CONFIG     True = reload the workbook into the config + member tables
#   WORKBOOK        the authored workbook, resolved on the attached lakehouse
#   BATCH_ID        the passed-in batch_id, or a fresh one when none was supplied
#
# Captured by value, on purpose: cell 7's `%run olaf` rebinds the lowercase names, so anything
# read after it must already live in UPPER_CASE or in PARAMS.
OLAF_NB = "olaf"
LAKEHOUSE = lakehouse_name
AUTO_APPROVE = auto_approve
KEEP_UNMANAGED = keep_unmanaged
REBUILD = rebuild
LOAD_CONFIG = load_config
WORKBOOK = config_workbook
BATCH_ID = batch_id.strip() or str(uuid.uuid4())

# Setup override parameters
PARAMS = {
    "env": env,
    "batch_id": BATCH_ID,
    # Schema-qualified names for the four control tables. These are OLAF's own defaults; point them
    # wherever your estate keeps them — every stage below and the facade all read the same PARAMS,
    # so they cannot drift apart.
    "config_table": "olaf.onelake_security_config",
    "mapping_table": "olaf.onelake_security_mapping",
    "member_table": "olaf.onelake_security_member",
    "log_table": "olaf.onelake_security_log",
}

print(
    "batch_id",
    BATCH_ID,
    "(passed in)" if batch_id.strip() else "(generated)",
    "· AUTO_APPROVE",
    AUTO_APPROVE,
)


# ── output helpers ────────────────────────────────────────────────────────────
STAGE_ICON = {
    "setup": "🧱",
    "validate": "🔍",
    "generate": "🧬",
    "plan": "📋",
    "apply": "🚀",
    "rollback": "↩️",
    "trace": "🔭",
    "show": "👥",
    "log": "📜",
}
STATUS_BADGE = {"success": "✅", "skipped": "⏭️", "blocked": "🚫", "error": "❌"}


def _flatten(data, _prefix=""):  # envelope data -> (item, value) rows
    rows = []
    for k, v in (data or {}).items():
        if isinstance(v, dict):
            rows += [(f"{_prefix}{k}.{k2}", str(v2)) for k2, v2 in v.items()] or [
                (f"{_prefix}{k}", "(none)")
            ]
        elif isinstance(v, (list, tuple)):
            rows.append((f"{_prefix}{k}", ", ".join(map(str, v)) if v else "(none)"))
        else:
            rows.append((f"{_prefix}{k}", str(v)))
    return rows


def report(stage, env, detail=True):  # header + message + data table
    icon, status = STAGE_ICON.get(stage, "•"), env.get("status", "?")
    print(f"{icon}  {stage.upper():9} {STATUS_BADGE.get(status, '•')} {status}")
    print(f"    {env.get('message', '')}")
    rows = _flatten(env.get("data")) if detail else []
    if rows:
        display(spark.createDataFrame(rows, "item STRING, value STRING"))
    return env


def role_actions(env, key="plan"):  # per-role verdicts, omissions first
    d = (env.get("data") or {}).get(key) or {}
    if not d:
        return
    # `omit`, not `delete`: a role left out of the payload is an omission candidate. The
    # runtime has never emitted "delete" here, so an order keyed on it sorted omissions last.
    order = {"omit": 0, "update": 1, "create": 2}
    rows = sorted(((r, a) for r, a in d.items()), key=lambda x: (order.get(x[1], 9), x[0]))
    print(f"    {key} — {len(rows)} role(s), omissions first")
    display(spark.createDataFrame(rows, "role STRING, action STRING"))


def fail_report(stage, exc):  # what the pipeline gets to see on a failure
    print(f"❌  {stage.upper():9} FAILED")
    print(f"    {str(exc)[:1200]}")
    try:  # validate writes no log row; the rest do
        rows = spark.table(PARAMS["log_table"]).where(f"batch_id = '{BATCH_ID}'").orderBy("run_at")
        display(rows) if rows.count() else print(
            "    (no log row — validate never logs; reason is above)"
        )
    except Exception as e:
        print("    (log unreadable:", e, ")")


def run_stage(stage, extra=None, detail=True):  # notebook.run + report · the CELL owns the try
    return report(
        stage,
        json.loads(
            notebookutils.notebook.run(OLAF_NB, 3600, {**PARAMS, "mode": stage, **(extra or {})})
        ),
        detail=detail,
    )


_RULE = re.compile(r"\s*\(rule (\w+)\)\s*$")  # MOST warnings end this way; cross-row
# and platform-limit warnings carry no "(rule XX)" suffix and land together under "?"
_ROW = re.compile(r"^row (\d+) \(([^)]*)\):\s*")  # per-row warnings name their role


def warnings_table(env):  # validate's warnings, one row each
    ws = (env.get("data") or {}).get("warnings") or []
    if not ws:
        print("    ✅ no warnings")
        return
    rows = []
    for w in ws:
        text = str(w)
        rule = _RULE.search(text)
        text = _RULE.sub("", text) if rule else text
        row = _ROW.match(text)
        rows.append(
            (
                rule.group(1) if rule else "?",
                row.group(1) if row else "",  # config row number, when it has one
                row.group(2) if row else "",  # the role it is about
                _ROW.sub("", text).strip(),
            )
        )
    by_rule = Counter(r[0] for r in rows)
    print(
        "    ⚠️  "
        + str(len(rows))
        + " warning(s) · "
        + " · ".join(f"{k} × {v}" for k, v in sorted(by_rule.items()))
    )
    # rarest rule first: one C4 among thirty B4s is the one worth reading
    rows.sort(key=lambda r: (by_rule[r[0]], r[0], int(r[1] or 0)))
    display(spark.createDataFrame(rows, "rule STRING, row STRING, subject STRING, warning STRING"))

## 1 · setup

Creates the control tables if they are missing. Idempotent, so it is safe on every run.


In [ ]:
try:
    if run_setup:
        setup = run_stage("setup", {"lakehouse_name": LAKEHOUSE})
    else:
        setup = None
        print("⏭️  SETUP     skipped — set run_setup = True after a framework upgrade")
except Exception as exc:
    fail_report("setup", exc)
    raise

## 1b · load the authored workbook  →  config + member

`load_config` is a **facade** call, not a mode, so the notebook loads `olaf` as a library first with
`%run`. That single `%run` serves this stage and the verify stage at the end.

> 🔴 **`OLAF.configure(**PARAMS)` is not optional, and it must come before any facade call.** `%run`
> brings in olaf's *own* parameter-cell defaults, which are not necessarily the tables this run uses.
> Without it, `load_config` would write to a different set of control tables than the stages below
> read. `configure` makes PARAMS sticky for every later facade call and hands back a frame of what is
> set, so the output is the receipt.
>
> `%run` also rebinds the loose lowercase names (`env`, `batch_id`, `config_table`, …). Everything
> this notebook needs afterwards is either `PARAMS` (a dict olaf never defines) or UPPER_CASE, which
> is why it survives — do not add lowercase state you expect to outlive this cell.
>
> **This is why a re-run starts at the top, not at the stage you want to redo.** After this cell,
> `env` is back to olaf's own default and `batch_id` is `""` — of the switches in the parameters
> cell, `env`, `batch_id` and `keep_unmanaged` are the three olaf also defines. Re-running from cell
> 3 onwards would silently log a `prod` run as `dev`, or turn a deliberate `keep_unmanaged = True`
> back into a destructive REPLACE, with no error either way. Likewise `LOAD_CONFIG` / `AUTO_APPROVE`
> are **captured** in cell 3: flipping the lowercase switch takes effect only once cell 3 re-runs.

A full REPLACE from the workbook: a row deleted there is deleted here. The sheet's columns must match
the table exactly — missing *and* unexpected are both refused, naming them, because a sheet loaded
with a column missing is a config that silently means something other than what the author edited.

No audit row is written; the Delta commit is the record. This is authoring, not deployment.

In [ ]:
%run olaf

In [ ]:
# same PARAMS as every stage · verbosity="quiet" drops olaf's result-key dump
try:
    print("⚙️   CONFIGURE  facade pointed at this run's control tables")
    display(OLAF.configure(**PARAMS, verbosity="quiet"))
except Exception as exc:
    fail_report("configure", exc)
    raise

try:
    if LOAD_CONFIG:
        print(f"📥  LOAD      {WORKBOOK} → config + member (full REPLACE)")
        display(OLAF.load_config("config", WORKBOOK, sheet="config"))
        display(OLAF.load_config("member", WORKBOOK, sheet="member"))
    else:
        print("⏭️  LOAD      skipped — set load_config = True and re-run from the top")
except Exception as exc:
    fail_report("load_config", exc)
    raise

## 2 · validate

Runs every rule `generate` runs, against the config, and writes **nothing** — no mapping, no CSV,
not even a log row. So a config that is going to be refused is refused here, before anything has
been committed, and the existing lock-file is left exactly as it was.

It also collects *all* failures rather than stopping at the first, so one run lists everything
wrong with the config.


In [ ]:
try:
    validate = run_stage("validate")
    warnings_table(validate)
except Exception as exc:
    fail_report("validate", exc)
    raise

## 3 · generate

Now that the config is known good, commit it: resolve it into the mapping lock-file that `plan` and
`apply` read, and export the versioned CSV to the mapping-history folder.

`skipped` means the config hash has not changed and the existing lock-file is still current — that
is a pass, not a problem.

**`rebuild` defaults to `False`, and that is an access decision, not a performance one.** The skip
is keyed on `config_hash`, which fingerprints the config rows only — it cannot see the live
catalog. So a table that appeared since the last generate and matches a wildcard (`sales.*`) is
**not** in the lock-file, and nothing is granted on it: a table created by last night's load does
not become readable because a star matched it. Opening it up is a decision somebody makes by
passing `rebuild = True` — which is all-or-nothing, re-resolving *every* wildcard, so it takes in
**every** table that has appeared since the last generate. Note the converse too: a table that was
**dropped** stays in the lock-file until the next real generate.


In [ ]:
try:
    generate = run_stage("generate", {"rebuild": REBUILD})

    if generate["status"] == "skipped":
        print("    → config unchanged, reusing the current lock-file")
except Exception as exc:
    fail_report("generate", exc)
    raise

## 4 · plan

Diffs the mapping against the live roles and writes the plan to the log. **No live change.**

`changed` is the branch: no drift means the deployment is already correct, so apply never runs and
nobody is asked to approve an empty plan.


In [ ]:
try:
    plan = run_stage("plan", detail=False)
    role_actions(plan, "plan")

    print(
        "\n    → changes to apply — approval required"
        if plan["changed"]
        else "\n    ✅ no drift — nothing to apply, the run ends here"
    )
except Exception as exc:
    fail_report("plan", exc)
    raise

## 5 · gate → apply

The only human step. In ADF this is an approval check; here it is `AUTO_APPROVE`.

**What actually binds apply to the plan is `config_hash`, not `batch_id`.** `apply` refuses unless
the log holds a successful `plan` row for this env and this same config hash, *and* the live state
still matches what that plan diffed against — edit the config between the two and apply refuses
rather than deploying a plan nobody reviewed. `batch_id` only correlates the run's log rows: it is
not in the gate's filter, so a shared batch_id neither grants nor withholds anything.


In [ ]:
apply_result = None  # bound in every branch, so the exit cell can read it safely

try:
    if not plan["changed"]:
        print("⏭️  APPLY     skipped — nothing to apply")

    elif not AUTO_APPROVE:
        print("⏸️  APPLY     held — set auto_approve = True and re-run from the top to apply")

    else:
        apply_result = run_stage("apply", {"keep_unmanaged": KEEP_UNMANAGED}, detail=False)
        # apply's envelope carries no per-role map, so there is no role_actions() call here: the
        # plan cell above already printed the verdicts, and what apply adds is what it actually did.
        # (An `applied` key existed once, renamed `push_status` because it holds the bulk PUT's
        # HTTP 200 -- "applied: 200" reads as 200 roles to anyone skimming an incident.)
        done = apply_result.get("data") or {}
        wrote, http = done.get("roles_written", "?"), done.get("push_status", "?")
        print(f"    ✍️  wrote {wrote} role(s) · HTTP {http}")
        gone = done.get("omitted_role_candidates") or []
        # CANDIDATES, not confirmed deletions: the Preview bulk endpoint does not document
        # deletion-by-omission, so what OLAF can say is which roles it left out of the payload.
        print(f"    ➖ omitted {len(gone)}:", ", ".join(gone) if gone else "none")
        if gone:
            print(
                "       ↳ omission is a REQUEST shape, not a confirmed outcome — check the post-state"
            )
        print("    💾 backup:", done.get("backup_path", "—"))
except Exception as exc:
    fail_report("apply", exc)
    raise

## 6 · verify — read the live state back

The `OLAF` facade is already bound and configured — stage 1b did both. The facade is the interactive
surface: unlike the pipeline path it **never raises on outcome** and hands back a DataFrame, so this
cell is safe to re-run and safe to read.

This is what turns "apply returned success" into "the live role set looks like this".


In [ ]:
try:
    print("🔭  TRACE     operational snapshot — the current generation end to end")
    display(OLAF.trace())

    print("👥  SHOW      the live role set, as it stands after the apply above")
    display(OLAF.show(by="role", subject="*"))  # subject is REQUIRED · "*" = every role
except Exception as exc:
    fail_report("verify", exc)
    raise

## What the run left behind

`olaf.onelake_security_log` keeps every row this run wrote, keyed by `batch_id` — the durable audit
trail, readable long after this session is gone.

The run timestamp column is `run_at` — not `logged_at`. The full column list is in [data-model.md](../docs/data-model.md).


In [ ]:
try:
    print(f"📜  LOG       every row this run wrote · batch_id {BATCH_ID}")
    display(spark.table(PARAMS["log_table"]).where(f"batch_id = '{BATCH_ID}'").orderBy("run_at"))
except Exception as exc:
    fail_report("log", exc)
    raise

## Recovery — two mechanisms, and they are not interchangeable

`apply` defaults to config-as-whole-truth: it omits every live role the config does not declare from the submitted payload, **including
`Default*`**. There are two ways back, and picking the wrong one loses roles.

| What went wrong | Use | Why |
|---|---|---|
| **apply failed part-way** — the push threw | the **pre-apply backup file** | it restores the live role set *exactly as it was*, including roles the config never declared |
| **apply succeeded, the config was wrong** | `rollback` | it restores a prior config version and re-runs generate → plan → apply |

> 🔴 **`rollback` is not a substitute for the backup.** It replays *config*, so it cannot rebuild
> roles config never declared — and that is precisely the set a default `apply` leaves out. After a
> failed or mistaken run, the backup file is the recovery input for putting `Default*` back; it is a
> recovery *input*, not a guaranteed exact restoration of platform state.

**The backup is automatic.** `apply` writes the live roles to `Files/security/role-backups/` before
it pushes, on every run, and **a failed backup aborts the apply** — nothing is pushed without a
restore point behind it. The path is printed by the apply cell above and carried in the envelope as
`data.backup_path`. Restoring is `json.load` → `put_roles`, nothing to edit in between
(RUNBOOK §3c).

**A failed push is not silent either.** Before the exception leaves `apply`, the framework re-reads
the live roles and writes a forensic record to the log: the per-grant rows re-stamped `failed` (so a
push that wrote nothing cannot later claim it wrote everything) and one row per planned role saying
whether it is `PRESENT` or `ABSENT` live. Read it with the log cell above before deciding anything.

Rollback is left commented below on purpose — an escape hatch, not part of the routine flow.


In [ ]:
# Uncomment to roll back. rollback_to_version "" = the version immediately before the current one.
# try:
#     rb = run_stage("rollback", {"rollback_to_version": "",
#                                 "rollback_reason": "why this deployment is being withdrawn"})
# except Exception as exc:
#     fail_report("rollback", exc)
#     raise

## Hand the result back to the caller

`notebook.exit` ends the notebook and returns this payload as the activity's `exitValue`, so the
pipeline that called this one can branch on whether anything actually changed instead of
re-querying the log.

Failures never reach here: a failed stage raises, the notebook fails, and the calling activity takes
its own **Failure** path.

**Keep this cell last.** `notebook.exit` stops execution, so anything below it would never run.


In [ ]:
try:
    result = {
        "batch_id": BATCH_ID,
        "changed": plan["changed"],  # did plan find drift to apply
        "applied": apply_result is not None,  # did apply actually run
        "backup_path": (apply_result or {}).get("data", {}).get("backup_path"),
    }
    print("🏁  DONE      returning to the caller")
    display(
        spark.createDataFrame([(k, str(v)) for k, v in result.items()], "item STRING, value STRING")
    )
except Exception as exc:
    fail_report("done", exc)
    raise

notebookutils.notebook.exit(json.dumps(result))